# HighRes-Net Training on Microscopy Data

This notebook trains HighRes-Net on your TIFF microscopy dataset (7 LR frames → 2x HR upsampling).

**Format:**
- ✓ TIFF 8-bit grayscale files (LR_1.tif/tiff through LR_7.tif/tiff + HR.tif/tiff per image)
- ✓ Pre-augmented images in D:\GUC\Datasets\HighRes input test\image_0000\, image_0001\, etc.
- ✓ 2x upsampling (128×128 LR → 256×256 HR)
- ✓ Supports both .tif and .tiff extensions (identical format)
- ✓ GPU acceleration with RTX 4060

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import time

sys.path.insert(0, '../src')

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from DataLoader import TiffPatchDataset, collateFunction
from DeepNetworks.HRNet import HRNet

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Verify Dataset

In [ ]:
data_root = Path("D:\\GUC\\Datasets\\HighRes input test")
image_dirs = sorted([d for d in data_root.iterdir() if d.is_dir() and d.name.startswith("image_")])

print(f"✓ Found {len(image_dirs)} images")

# Verify first image has required files
if image_dirs:
    first = image_dirs[0]
    lr_files = sorted(first.glob("LR_*.tif*"))
    hr_file = list(first.glob("HR.tif*"))
    print(f"✓ Sample ({first.name}): {len(lr_files)} LR frames, {len(hr_file)} HR frame")

## 3. Load Configuration

In [ ]:
config_path = Path('../config/config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

print("Training Configuration:")
print(f"  Num epochs: {config['training']['num_epochs']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['lr']}")
print(f"  Patch size: {config['training']['patch_size']}")
print(f"  N_views: {config['training']['n_views']}")
print(f"  Create patches: {config['training']['create_patches']}")

## 5. Create Datasets and DataLoaders

In [ ]:
print("Creating TIFF dataset...")

# Discover all image directories
data_root = Path("D:\\GUC\\Datasets\\HighRes input test")
image_dirs = sorted([str(d) for d in data_root.iterdir() if d.is_dir() and d.name.startswith("image_")])

print(f"Discovered {len(image_dirs)} images")

# Create dataset from images
train_dataset = TiffPatchDataset(
    patch_dirs=image_dirs,
    max_views=config['training']['n_views']
)

print(f"Training dataset size: {len(train_dataset)} images")

# Create DataLoader
batch_size = config['training']['batch_size']
min_L = config['training']['min_L']

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=collateFunction(min_L=min_L),
    pin_memory=torch.cuda.is_available()
)

print(f"DataLoader created:")
print(f"  batch_size: {batch_size}")
print(f"  num_batches: {len(train_loader)}")
print(f"  pin_memory: {torch.cuda.is_available()}")

## 6. Initialize Model

In [ ]:
print("Initializing model...")

model = HRNet(config['network'])
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Model device: {device}")

## 7. Setup Optimizer and Loss

In [ ]:
print("Setting up training...")

criterion = torch.nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=config['training']['lr'],
    betas=(0.9, 0.999)
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=config['training']['lr_step'],
    gamma=config['training']['lr_decay']
)

print(f"Optimizer: Adam")
print(f"  Learning rate: {config['training']['lr']}")
print(f"  LR decay: {config['training']['lr_decay']} every {config['training']['lr_step']} epochs")
print(f"Loss function: MSELoss")

## 8. Training Loop

In [ ]:
# Training weight manager (selection + organization)
import json
import shutil
from datetime import datetime
from pathlib import Path

WEIGHTS_ROOT = Path("../models/weights")
BASE_LIBRARY_DIR = WEIGHTS_ROOT / "Base"
CBAM_LIBRARY_DIR = WEIGHTS_ROOT / "CBAM"
RUNS_TMP_DIR = WEIGHTS_ROOT / "_runs_tmp"
REGISTRY_PATH = WEIGHTS_ROOT / "weights_registry.json"
ACTIVE_SELECTION_PATH = WEIGHTS_ROOT / "active_weight_selection.json"
LAST_TRAINING_CANDIDATE_PATH = WEIGHTS_ROOT / "last_training_candidate.json"

for folder in [WEIGHTS_ROOT, BASE_LIBRARY_DIR, CBAM_LIBRARY_DIR, RUNS_TMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def _load_registry(path: Path):
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return {"weights": []}


def _save_registry(path: Path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


def _safe_score(meta):
    val = meta.get("best_score", "")
    return val if val else "-"


def _collect_library_entries(root_dir: Path, family: str):
    entries = []
    for p in sorted(root_dir.glob("*.pth"), key=lambda x: x.stat().st_mtime, reverse=True):
        sidecar = p.with_suffix(".json")
        meta = {}
        if sidecar.exists():
            try:
                with open(sidecar, "r") as f:
                    meta = json.load(f)
            except Exception:
                meta = {}
        entries.append({
            "family": family,
            "name": p.stem,
            "path": str(p.resolve()),
            "mtime": datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
            "size_mb": p.stat().st_size / 1e6,
            "best_score": _safe_score(meta),
            "notes": meta.get("notes", "")
        })
    return entries


def list_weight_library():
    base_entries = _collect_library_entries(BASE_LIBRARY_DIR, "Base")
    cbam_entries = _collect_library_entries(CBAM_LIBRARY_DIR, "CBAM")

    all_entries = []
    print("=" * 90)
    print("WEIGHT LIBRARY")
    print("=" * 90)
    print("[Base]")
    if not base_entries:
        print("  (empty)")
    for _, e in enumerate(base_entries):
        all_entries.append(e)
        print(f"  {len(all_entries)-1:2d} | {e['name'][:35]:35s} | {e['best_score']:8s} | {e['size_mb']:6.2f} MB | {e['mtime']}")

    print("\n[CBAM]")
    if not cbam_entries:
        print("  (empty)")
    for _, e in enumerate(cbam_entries):
        all_entries.append(e)
        print(f"  {len(all_entries)-1:2d} | {e['name'][:35]:35s} | {e['best_score']:8s} | {e['size_mb']:6.2f} MB | {e['mtime']}")

    print("=" * 90)
    return all_entries


def promote_weight_to_library(source_weight_path: Path, destination_family: str, score_label: str, notes: str, custom_name: str = ""):
    destination_dir = BASE_LIBRARY_DIR if destination_family == "Base" else CBAM_LIBRARY_DIR
    destination_dir.mkdir(parents=True, exist_ok=True)

    if custom_name.strip():
        output_name = custom_name.strip().replace(" ", "_")
    else:
        label_part = f"_{score_label}" if score_label else ""
        output_name = f"{run_stamp}_{destination_family.lower()}_{run_init}_{RUN_TAG}{label_part}"

    target_weight_path = destination_dir / f"{output_name}.pth"

    suffix = 2
    while target_weight_path.exists():
        target_weight_path = destination_dir / f"{output_name}_v{suffix}.pth"
        suffix += 1

    shutil.copy2(source_weight_path, target_weight_path)

    sidecar = {
        "name": target_weight_path.stem,
        "family": destination_family,
        "source_weight": str(selected_source_weight_path) if selected_source_weight_path else None,
        "run_name": run_name,
        "run_dir": str(run_dir),
        "created_at": datetime.now().isoformat(),
        "best_score": score_label,
        "notes": notes,
        "best_training_loss": float(best_loss) if 'best_loss' in globals() else None
    }
    with open(target_weight_path.with_suffix(".json"), "w") as f:
        json.dump(sidecar, f, indent=2)

    registry = _load_registry(REGISTRY_PATH)
    registry["weights"] = [w for w in registry.get("weights", []) if w.get("path") != str(target_weight_path.resolve())]
    registry["weights"].append({
        "name": target_weight_path.stem,
        "family": destination_family,
        "path": str(target_weight_path.resolve()),
        "best_score": score_label,
        "notes": notes,
        "source_weight": str(selected_source_weight_path) if selected_source_weight_path else None,
        "run_name": run_name,
        "created_at": datetime.now().isoformat()
    })
    _save_registry(REGISTRY_PATH, registry)
    return target_weight_path


def _read_active_selection():
    if not ACTIVE_SELECTION_PATH.exists():
        return None
    try:
        with open(ACTIVE_SELECTION_PATH, "r") as f:
            data = json.load(f)
        path_value = data.get("path")
        if not path_value:
            return None
        candidate = Path(path_value)
        if candidate.exists():
            return candidate
    except Exception:
        return None
    return None


# ---------------------------------------------------------------------------
# USER CONTROLS (edit these only)
# ---------------------------------------------------------------------------
ACTIVE_ARCH_FAMILY = "Base"      # "Base" or "CBAM" (default library destination)
SOURCE_WEIGHT_INDEX = None        # Optional manual override index from printed table. None = use active-selection or scratch.
USE_ACTIVE_SELECTION_FILE = True  # If True and SOURCE_WEIGHT_INDEX is None, use models/weights/active_weight_selection.json
RUN_TAG = "coil_experiment"

library_entries = list_weight_library()
selected_source_weight_path = None
selected_source_weight_name = None

if SOURCE_WEIGHT_INDEX is not None:
    if SOURCE_WEIGHT_INDEX < 0 or SOURCE_WEIGHT_INDEX >= len(library_entries):
        raise ValueError(f"Invalid SOURCE_WEIGHT_INDEX={SOURCE_WEIGHT_INDEX}. Select from listed table.")
    selected = library_entries[SOURCE_WEIGHT_INDEX]
    selected_source_weight_path = Path(selected["path"])
    selected_source_weight_name = selected["name"]
elif USE_ACTIVE_SELECTION_FILE:
    selected_source_weight_path = _read_active_selection()
    if selected_source_weight_path is not None:
        selected_source_weight_name = selected_source_weight_path.stem

if selected_source_weight_path is not None:
    print(f"Selected source weight: {selected_source_weight_path}")
else:
    print("No source weight selected -> training from scratch")

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_init = "finetune" if selected_source_weight_path else "scratch"
run_name = f"{run_stamp}_{ACTIVE_ARCH_FAMILY.lower()}_{run_init}_{RUN_TAG}"
run_dir = RUNS_TMP_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)

managed_best_weights_path = run_dir / "best.pth"
managed_checkpoint_dir = run_dir / "checkpoints"
managed_checkpoint_dir.mkdir(parents=True, exist_ok=True)
managed_checkpoint_meta_path = run_dir / "training_metadata.json"

print("\nRun setup:")
print(f"  run_name: {run_name}")
print(f"  run_dir: {run_dir}")
print(f"  active_arch_family: {ACTIVE_ARCH_FAMILY}")

# Keep legacy helpers available for manual rollback if needed
legacy_weights_dir = WEIGHTS_ROOT
legacy_checkpoints_dir = WEIGHTS_ROOT / "checkpoints"
legacy_backups_dir = WEIGHTS_ROOT / "backups"
legacy_best_weights_path = WEIGHTS_ROOT / "HRNet.pth"
legacy_checkpoints_dir.mkdir(parents=True, exist_ok=True)
legacy_backups_dir.mkdir(parents=True, exist_ok=True)


def create_manual_backup(tag="manual"):
    if not legacy_best_weights_path.exists():
        print(f"No current best file at {legacy_best_weights_path}; nothing to back up.")
        return None

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = legacy_backups_dir / f"HRNet_backup_{tag}_{stamp}.pth"
    shutil.copy2(legacy_best_weights_path, backup_path)
    print(f"Backup created: {backup_path}")
    return backup_path


def restore_weights_from(path_or_name):
    path_or_name = str(path_or_name)
    candidate = Path(path_or_name)

    if not candidate.exists():
        for base in [legacy_weights_dir, legacy_checkpoints_dir, legacy_backups_dir, BASE_LIBRARY_DIR, CBAM_LIBRARY_DIR]:
            probe = base / path_or_name
            if probe.exists():
                candidate = probe
                break

    if not candidate.exists():
        raise FileNotFoundError(f"Could not find weights file: {path_or_name}")

    shutil.copy2(candidate, legacy_best_weights_path)
    print(f"Restored {candidate} -> {legacy_best_weights_path}")
    return legacy_best_weights_path


print("Weight manager ready.")
print("Use notebooks/weights_manager.ipynb to select active weight and to save/discard after training.")

In [ ]:
from tqdm import tqdm
import json as json_module
import sys
import shutil
from datetime import datetime

# ============================================================================
# Range + smoothness regularizers
# ============================================================================
def compute_range_loss(sr_output, hr_target):
    """
    Prevent output dynamic-range collapse by matching SR/HR range statistics.
    """
    sr_min = sr_output.min()
    sr_max = sr_output.max()
    sr_range = sr_max - sr_min

    hr_min = hr_target.min()
    hr_max = hr_target.max()

    target_range = 0.6
    range_loss = (sr_range - target_range) ** 2
    min_align = (sr_min - hr_min) ** 2
    max_align = (sr_max - hr_max) ** 2
    return range_loss * 0.5 + (min_align + max_align) * 0.25


def tv_loss(output):
    """Standard TV: penalizes all local variation (can oversmooth edges)."""
    diff_h = torch.abs(output[:, :, 1:, :] - output[:, :, :-1, :])
    diff_w = torch.abs(output[:, :, :, 1:] - output[:, :, :, :-1])
    return torch.mean(diff_h) + torch.mean(diff_w)


def edge_aware_tv_loss(output, hr_target, edge_k=10.0):
    """
    Penalize SR variation mostly where HR is smooth, and relax penalty at true HR edges.
    """
    hr_grad_h = torch.abs(hr_target[:, :, 1:, :] - hr_target[:, :, :-1, :])
    hr_grad_w = torch.abs(hr_target[:, :, :, 1:] - hr_target[:, :, :, :-1])

    edge_weight_h = torch.exp(-edge_k * hr_grad_h)
    edge_weight_w = torch.exp(-edge_k * hr_grad_w)

    sr_grad_h = torch.abs(output[:, :, 1:, :] - output[:, :, :-1, :])
    sr_grad_w = torch.abs(output[:, :, :, 1:] - output[:, :, :, :-1])

    return torch.mean(edge_weight_h * sr_grad_h) + torch.mean(edge_weight_w * sr_grad_w)


def _radial_frequency_mask(h, w, low_ratio, high_ratio, device, dtype):
    """Build a radial band-pass mask in normalized FFT frequency radius [0, 1]."""
    fy = torch.fft.fftfreq(h, d=1.0, device=device).view(h, 1)
    fx = torch.fft.fftfreq(w, d=1.0, device=device).view(1, w)
    r = torch.sqrt(fy * fy + fx * fx)
    r = r / r.max().clamp_min(1e-8)
    mask = ((r >= low_ratio) & (r <= high_ratio)).to(dtype)
    return mask


def freq_band_loss(sr_output, hr_target, low_ratio=0.08, high_ratio=0.35, use_log_magnitude=True):
    """
    Penalize spectral error energy in a selected mid-frequency band.
    This is more targeted than spatial TV for ringing suppression.
    """
    err = sr_output - hr_target
    err_fft = torch.fft.fft2(err, dim=(-2, -1))
    err_mag = torch.abs(err_fft)

    if use_log_magnitude:
        err_mag = torch.log1p(err_mag)

    h = err_mag.shape[-2]
    w = err_mag.shape[-1]
    band = _radial_frequency_mask(h, w, low_ratio, high_ratio, err_mag.device, err_mag.dtype)
    band = band.unsqueeze(0).unsqueeze(0)

    denom = band.sum() * err_mag.shape[0] * err_mag.shape[1]
    return ((err_mag * band) ** 2).sum() / denom.clamp_min(1.0)


print("\nStarting training with configurable composite loss...")
print("=" * 70)
print("Loss = MSE + lambda_range * RangeLoss + lambda_smooth * SmoothnessLoss")
print("SmoothnessLoss mode: none | tv | edge_aware_tv | freq_band")
print("=" * 70)

num_epochs = config['training']['num_epochs']
lambda_range = config['training'].get('lambda_range', 0.003)

smoothness_mode = config['training'].get('smoothness_mode', 'tv').lower()
lambda_tv = config['training'].get('lambda_tv', 0.0)
lambda_edge_tv = config['training'].get('lambda_edge_tv', 0.0)
lambda_freq = config['training'].get('lambda_freq', 0.0)
edge_aware_k = float(config['training'].get('edge_aware_k', 10.0))
freq_low_ratio = float(config['training'].get('freq_low_ratio', 0.08))
freq_high_ratio = float(config['training'].get('freq_high_ratio', 0.35))
freq_use_log_magnitude = bool(config['training'].get('freq_use_log_magnitude', True))

if smoothness_mode == 'edge_aware_tv':
    lambda_smooth = lambda_edge_tv if lambda_edge_tv > 0 else lambda_tv
elif smoothness_mode == 'tv':
    lambda_smooth = lambda_tv
elif smoothness_mode == 'freq_band':
    lambda_smooth = lambda_freq
else:
    lambda_smooth = 0.0

best_loss = float('inf')
best_weights_path = managed_best_weights_path
checkpoint_meta_path = managed_checkpoint_meta_path
checkpoint_dir = managed_checkpoint_dir

# Optional source load: if selected, this is fine-tuning. If not selected, scratch.
if selected_source_weight_path is not None:
    print(f"\nLoading source weights for fine-tuning: {selected_source_weight_path}")
    source_state_dict = torch.load(selected_source_weight_path, map_location=device)
    missing_keys, unexpected_keys = model.load_state_dict(source_state_dict, strict=False)
    print(f"Source load complete. Missing keys: {len(missing_keys)}, Unexpected keys: {len(unexpected_keys)}")
    if missing_keys:
        print("  Missing keys sample:", missing_keys[:5])
    if unexpected_keys:
        print("  Unexpected keys sample:", unexpected_keys[:5])
else:
    print("\nNo source weights selected. Training from scratch.")

print(f"\nlambda_range:            {lambda_range}")
print(f"smoothness_mode:         {smoothness_mode}")
print(f"lambda_smooth:           {lambda_smooth}")
print(f"edge_aware_k:            {edge_aware_k}")
print(f"freq_low/high ratios:    {freq_low_ratio} / {freq_high_ratio}")
print(f"freq_use_log_magnitude:  {freq_use_log_magnitude}")
print(f"best_weights_path:       {best_weights_path}")
print(f"checkpoint_dir:          {checkpoint_dir}\n")

train_losses = []
train_mse_losses = []
train_range_losses = []
train_tv_losses = []
train_edge_tv_losses = []
train_freq_losses = []
epoch_times = []

pbar = tqdm(
    range(1, num_epochs + 1),
    desc="Training",
    unit="epoch",
    file=sys.stdout,
    ncols=100,
    disable=False,
    bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
)

for epoch in pbar:
    epoch_start = time.time()

    model.train()
    batch_losses = []
    batch_mse_losses = []
    batch_range_losses = []
    batch_tv_losses = []
    batch_edge_tv_losses = []
    batch_freq_losses = []

    for batch_idx, (lrs, alphas, hrs, hr_maps, names) in enumerate(train_loader):
        lrs = lrs.float().to(device)
        alphas = alphas.float().to(device)
        hrs = hrs.float().to(device)

        sr_output = model(lrs, alphas)

        if epoch == 1 and batch_idx == 0:
            print("\nShape check:")
            print(f"  LR input shape: {lrs.shape}")
            print(f"  SR output shape: {sr_output.shape}")
            print(f"  HR target shape: {hrs.shape}")

        mse_loss = criterion(sr_output, hrs)
        range_loss = compute_range_loss(sr_output, hrs)
        tv_reg = tv_loss(sr_output)
        edge_tv_reg = edge_aware_tv_loss(sr_output, hrs, edge_k=edge_aware_k)
        freq_reg = freq_band_loss(
            sr_output,
            hrs,
            low_ratio=freq_low_ratio,
            high_ratio=freq_high_ratio,
            use_log_magnitude=freq_use_log_magnitude,
        )

        if smoothness_mode == 'edge_aware_tv':
            smooth_loss = edge_tv_reg
        elif smoothness_mode == 'tv':
            smooth_loss = tv_reg
        elif smoothness_mode == 'freq_band':
            smooth_loss = freq_reg
        else:
            smooth_loss = torch.zeros_like(mse_loss)

        loss = mse_loss + lambda_range * range_loss + lambda_smooth * smooth_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())
        batch_mse_losses.append(mse_loss.item())
        batch_range_losses.append(range_loss.item())
        batch_tv_losses.append(tv_reg.item())
        batch_edge_tv_losses.append(edge_tv_reg.item())
        batch_freq_losses.append(freq_reg.item())

    epoch_loss = np.mean(batch_losses)
    epoch_mse_loss = np.mean(batch_mse_losses)
    epoch_range_loss = np.mean(batch_range_losses)
    epoch_tv_loss = np.mean(batch_tv_losses)
    epoch_edge_tv_loss = np.mean(batch_edge_tv_losses)
    epoch_freq_loss = np.mean(batch_freq_losses)

    train_losses.append(epoch_loss)
    train_mse_losses.append(epoch_mse_loss)
    train_range_losses.append(epoch_range_loss)
    train_tv_losses.append(epoch_tv_loss)
    train_edge_tv_losses.append(epoch_edge_tv_loss)
    train_freq_losses.append(epoch_freq_loss)

    scheduler.step()

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)

    pbar.set_postfix({
        'loss': f'{epoch_loss:.6f}',
        'mse': f'{epoch_mse_loss:.6f}',
        'range': f'{epoch_range_loss:.4f}',
        'tv': f'{epoch_tv_loss:.4f}',
        'e_tv': f'{epoch_edge_tv_loss:.4f}',
        'freq': f'{epoch_freq_loss:.4f}',
        'time': f'{epoch_time:.1f}s'
    })

    sys.stdout.flush()

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), best_weights_path)
        pbar.write(f"  Best loss updated at epoch {epoch}")

    if epoch % 25 == 0:
        checkpoint_path = checkpoint_dir / f"checkpoint_epoch_{epoch}.pth"
        torch.save(model.state_dict(), checkpoint_path)
        pbar.write(f"  Checkpoint saved: epoch_{epoch}.pth")

    if epoch % 10 == 0:
        meta = {
            'run_name': run_name,
            'source_weight': str(selected_source_weight_path) if selected_source_weight_path else None,
            'last_saved_epoch': epoch,
            'best_loss': float(best_loss),
            'current_loss': float(epoch_loss),
            'current_mse': float(epoch_mse_loss),
            'current_range': float(epoch_range_loss),
            'current_tv': float(epoch_tv_loss),
            'current_edge_tv': float(epoch_edge_tv_loss),
            'current_freq': float(epoch_freq_loss),
            'smoothness_mode': smoothness_mode,
            'lambda_smooth': float(lambda_smooth),
            'edge_aware_k': float(edge_aware_k),
            'freq_low_ratio': float(freq_low_ratio),
            'freq_high_ratio': float(freq_high_ratio),
            'freq_use_log_magnitude': bool(freq_use_log_magnitude),
            'num_epochs': num_epochs,
            'dataset_size': len(train_dataset),
            'timestamp': time.time()
        }
        with open(checkpoint_meta_path, 'w') as f:
            json_module.dump(meta, f, indent=2)

pbar.close()

print("\n" + "=" * 70)
print("Training complete")
print(f"Best combined loss: {best_loss:.6f}")
print(f"Run best weights: {best_weights_path}")
print(f"Run metadata: {checkpoint_meta_path}")
print(f"Run checkpoints: {checkpoint_dir}")
print("\nTraining Statistics:")
print(f"  Total time: {sum(epoch_times) / 60:.1f} minutes")
print(f"  Avg time per epoch: {np.mean(epoch_times):.1f}s")
print("\nMSE Loss (pixel accuracy):")
print(f"  Initial: {train_mse_losses[0]:.6f}")
print(f"  Final:   {train_mse_losses[-1]:.6f}")
print(f"  Improvement: {(train_mse_losses[0] - train_mse_losses[-1]) / train_mse_losses[0] * 100:.1f}%")
print("\nRange Loss (dynamic range):")
print(f"  Initial: {train_range_losses[0]:.6f}")
print(f"  Final:   {train_range_losses[-1]:.6f}")
print(f"  Improvement: {(train_range_losses[0] - train_range_losses[-1]) / train_range_losses[0] * 100:.1f}%")
print("\nSmoothness diagnostics:")
print(f"  TV initial/final:       {train_tv_losses[0]:.6f} -> {train_tv_losses[-1]:.6f}")
print(f"  Edge-TV initial/final:  {train_edge_tv_losses[0]:.6f} -> {train_edge_tv_losses[-1]:.6f}")
print(f"  Freq initial/final:     {train_freq_losses[0]:.6f} -> {train_freq_losses[-1]:.6f}")
print(f"  Active smoothness mode: {smoothness_mode} (lambda={lambda_smooth})")
print("=" * 70)

# Keep/discard is intentionally deferred to weights_manager.ipynb.
last_trained_weight_path = best_weights_path
last_saved_weight_path = None
post_training_promotion_done = False

candidate_summary = {
    "run_name": run_name,
    "run_dir": str(run_dir),
    "best_weight_path": str(last_trained_weight_path),
    "checkpoint_dir": str(checkpoint_dir),
    "meta_path": str(checkpoint_meta_path),
    "source_weight": str(selected_source_weight_path) if selected_source_weight_path else None,
    "arch_family": ACTIVE_ARCH_FAMILY,
    "created_at": datetime.now().isoformat(),
    "best_training_loss": float(best_loss)
}
with open(LAST_TRAINING_CANDIDATE_PATH, "w") as f:
    json.dump(candidate_summary, f, indent=2)

print("\nTraining finished. Review plots/diagnostics, then save/discard via notebooks/weights_manager.ipynb")
print(f"last_trained_weight_path: {last_trained_weight_path}")
print(f"candidate summary file:   {LAST_TRAINING_CANDIDATE_PATH}")

In [ ]:
from pathlib import Path

print("=" * 80)
print("TRAINING RUN SUMMARY")
print("=" * 80)

if 'last_trained_weight_path' in globals() and last_trained_weight_path is not None:
    print(f"Candidate trained weight: {last_trained_weight_path}")
else:
    print("Candidate trained weight: (not available - run training cell first)")

if 'run_dir' in globals():
    print(f"Run directory:           {run_dir}")

if 'LAST_TRAINING_CANDIDATE_PATH' in globals():
    print(f"Candidate summary file:  {LAST_TRAINING_CANDIDATE_PATH}")

print("\nNext step:")
print("  Open notebooks/weights_manager.ipynb")
print("  - List saved + temp weights")
print("  - Choose keep/discard for temp candidate")
print("  - Set active weight for next fine-tune or inference")

print("\nCurrent libraries:")
list_weight_library()
print("=" * 80)

## 9. Plot Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training loss
axes[0].plot(train_losses, linewidth=2, color='blue')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Epoch time
axes[1].plot(epoch_times, linewidth=2, color='orange')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Time (seconds)', fontsize=12)
axes[1].set_title('Epoch Computation Time', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining Statistics:")
print(f"  Total time: {sum(epoch_times)/60:.1f} minutes")
print(f"  Avg time per epoch: {np.mean(epoch_times):.1f}s")
print(f"  Loss decreased by: {(train_losses[0] - train_losses[-1]) / train_losses[0] * 100:.1f}%")

## 10. Test Trained Model

In [ ]:
# Load selected weight for quick sanity test in this notebook
from pathlib import Path
import torch

# If training ran in this session, prefer the most recent managed output.
# Otherwise, fallback to legacy HRNet.pth.
def _resolve_quick_test_weight():
    if 'last_saved_weight_path' in globals() and last_saved_weight_path is not None:
        candidate = Path(last_saved_weight_path)
        if candidate.exists():
            return candidate

    if 'last_trained_weight_path' in globals() and last_trained_weight_path is not None:
        candidate = Path(last_trained_weight_path)
        if candidate.exists():
            return candidate

    return Path("../models/weights/HRNet.pth")

selected_weights = _resolve_quick_test_weight()
if not selected_weights.exists():
    raise FileNotFoundError(f"Weight file not found: {selected_weights}")

state_dict = torch.load(selected_weights, map_location=device)
model.load_state_dict(state_dict, strict=False)
model.eval()

print(f"Loaded weights from: {selected_weights}")

In [ ]:
print("Testing trained model...\n")

# Uses weights already loaded in previous cell. If that cell was skipped, fallback to HRNet.pth
if 'selected_weights' not in globals():
    selected_weights = Path("../models/weights/HRNet.pth")
    model.load_state_dict(torch.load(selected_weights, map_location=device))
    model.eval()
    print(f"Loaded fallback weights: {selected_weights}")
else:
    print(f"Using preloaded weights: {selected_weights}")

# Get test batch
test_batch = next(iter(train_loader))
test_lrs, test_alphas, test_hrs, test_hr_maps, test_names = test_batch

test_lrs = test_lrs.float().to(device)
test_alphas = test_alphas.float().to(device)

# Forward pass
with torch.no_grad():
    test_sr = model(test_lrs, test_alphas)

print(f"✓ Test inference passed!")
print(f"  Input (LR):  {test_lrs.shape}")
print(f"  Output (SR): {test_sr.shape}")
print(f"  Output range: [{test_sr.min():.4f}, {test_sr.max():.4f}]")

print(f"\n✓ Active weights: {selected_weights}")
print(f"\nNext steps:")
print(f"  1. Run inference_diagnostic.ipynb to evaluate quality")
print(f"  2. Check PSNR/SSIM metrics vs bicubic baseline")
print(f"  3. If quality still low, train for more epochs or with more data")